# Kaggle Classical Baseline

This notebook trains and evaluates classical machine learning models on the Kaggle Human-LLM 
Generated Phishing/Legitimate Emails dataset. It uses TF-IDF vectorisation with Logistic 
Regression, Multinomial Naive Bayes, Linear SVC, and Random Forest classifiers. Results are 
evaluated overall and broken down by email source (human-generated vs LLM-generated) for 
comparison with the MeAJOR classical baseline.

Key methodological choices:
- `class_weight='balanced'` is applied to Logistic Regression and Linear SVC to handle the ~72/28 class imbalance.
- `class_weight='balanced_subsample'` is applied to Random Forest for the same reason.
- `GridSearchCV` is used for Logistic Regression and Linear SVC to ensure hyperparameters are properly optimised on training data only.

## 1. Import Libraries
Import all libraries needed for loading data, vectorising text, training classifiers, 
and evaluating results.

In [ ]:
# Standard library for working with file paths
from pathlib import Path
import time

# pandas for loading and manipulating tabular data
import pandas as pd

# numpy for numerical operations
import numpy as np

# TF-IDF converts raw text into numerical feature vectors
from sklearn.feature_extraction.text import TfidfVectorizer

# The four classical classifiers we will evaluate
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# GridSearchCV for hyperparameter tuning on training data only
from sklearn.model_selection import GridSearchCV

# Tools for evaluating model performance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

# matplotlib for plotting confusion matrices
import matplotlib.pyplot as plt

# warnings to suppress non-critical sklearn output
import warnings
warnings.filterwarnings('ignore')

## 2. Load Processed Data
Load the train and test Parquet files produced by `kaggle_preparation.ipynb`. These files 
contain the cleaned, deduplicated `text`, `label`, and `source` columns only.

In [ ]:
# Path to the folder where the processed Kaggle parquet files were saved
data_dir = Path('../data/processed/kaggle/')

# Load the 60% training split
train_df = pd.read_parquet(data_dir / 'kaggle_train_60.parquet')

# Load the 40% test split
test_df = pd.read_parquet(data_dir / 'kaggle_test_40.parquet')

print(f'Train size: {len(train_df)} rows')
print(f'Test size:  {len(test_df)} rows')
print(f'\nColumns: {train_df.columns.tolist()}')

## 3. Inspect Loaded Data
A quick check to confirm the data loaded correctly, the label distribution is preserved, 
and both sources are represented in each split.

In [ ]:
# Preview the first few rows to confirm structure looks correct
train_df.head(3)

In [ ]:
# Check label balance — training set is ~72% legitimate / ~28% phishing
print('Train label distribution:')
print(train_df['label'].value_counts())
print()
print('Test label distribution:')
print(test_df['label'].value_counts())

In [ ]:
# Confirm both human and LLM sources are present in each split
print('Train source breakdown:')
print(train_df['source'].value_counts())
print()
print('Test source breakdown:')
print(test_df['source'].value_counts())

## 4. Prepare Features and Labels
Separate the text (features) from the label (target) for both splits. The `source` column 
is kept separately for the per-source evaluation in Section 7.

In [ ]:
# X = the input text the model learns from; y = the label it tries to predict
X_train = train_df['text']
y_train = train_df['label']

X_test = test_df['text']
y_test = test_df['label']

# Keep source column aside for breakdown analysis later
source_test = test_df['source']

print(f'X_train: {X_train.shape[0]} samples')
print(f'X_test:  {X_test.shape[0]} samples')

## 5. TF-IDF Vectorisation
TF-IDF converts each email's text into a numerical vector. The vectoriser is fitted on 
training data only (fit_transform), then applied to the test set (transform only) 
to prevent data leakage.

In [ ]:
# max_features limits the vocabulary to the 50,000 most frequent terms
# ngram_range=(1,2) includes both single words and two-word phrases
# sublinear_tf applies log scaling to term frequencies
vectoriser = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)

# fit_transform learns the vocabulary from training data ONLY and transforms it
X_train_tfidf = vectoriser.fit_transform(X_train)

# transform applies the already-fitted vocabulary to the test data — no fitting here
X_test_tfidf = vectoriser.transform(X_test)

print(f'TF-IDF matrix shape (train): {X_train_tfidf.shape}')
print(f'TF-IDF matrix shape (test):  {X_test_tfidf.shape}')

## 6. Define Models
Four classical classifiers are defined here:

- **MultinomialNB**: No class_weight support; works naturally with TF-IDF probabilities.
- **Logistic Regression**: `class_weight='balanced'` corrects for the ~72/28 imbalance. Tuned via GridSearchCV.
- **Linear SVC**: `class_weight='balanced'` for the same reason. Tuned via GridSearchCV.
- **Random Forest**: `class_weight='balanced_subsample'` applies balanced weighting per bootstrap sample.

In [ ]:
# --- Logistic Regression: tune C via GridSearchCV on training data only ---
lr_base = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_grid = GridSearchCV(lr_base, param_grid={'C': [0.01, 0.1, 1, 10]}, cv=5, scoring='f1', n_jobs=-1)
lr_grid.fit(X_train_tfidf, y_train)
print(f'Best LR params: {lr_grid.best_params_}')

# --- Linear SVC: tune C via GridSearchCV on training data only ---
svc_base = LinearSVC(max_iter=2000, random_state=42, class_weight='balanced')
svc_grid = GridSearchCV(svc_base, param_grid={'C': [0.01, 0.1, 1, 10]}, cv=5, scoring='f1', n_jobs=-1)
svc_grid.fit(X_train_tfidf, y_train)
print(f'Best SVC params: {svc_grid.best_params_}')

# --- Multinomial NB: no class_weight, no tuning needed ---
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

# --- Random Forest: balanced_subsample applies weighting per bootstrap tree ---
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced_subsample', n_jobs=-1)
rf_model.fit(X_train_tfidf, y_train)

# Collect final fitted models for evaluation
models = {
    'TF-IDF + MultinomialNB':       nb_model,
    'TF-IDF + LogisticRegression':  lr_grid.best_estimator_,
    'TF-IDF + LinearSVM':           svc_grid.best_estimator_,
    'TF-IDF + RandomForest':        rf_model,
}

## 7. Evaluate All Models
Each model is evaluated on the held-out test set. Metrics reported: Accuracy, Precision, 
Recall, F1, False Positive Rate, and Inference Time — consistent with the unified evaluation 
framework used across all models in this project.

In [ ]:
# Create the figures folder for saving confusion matrix images
figures_dir = Path('../results/figures/')
figures_dir.mkdir(parents=True, exist_ok=True)

all_predictions = {}
all_results     = []

for name, model in models.items():
    print(f"\n{'='*55}")
    print(f'  {name}')
    print(f"{'='*55}")

    # Time inference on the test set
    start = time.perf_counter()
    y_pred = model.predict(X_test_tfidf)
    elapsed = time.perf_counter() - start

    all_predictions[name] = y_pred

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    print(classification_report(y_test, y_pred, target_names=['Legitimate (0)', 'Phishing (1)']))
    print(f'  False Positive Rate : {fpr:.4f}')
    print(f'  Inference Time      : {elapsed / len(y_test) * 1000:.4f} ms/email')

    all_results.append({
        'Dataset':    'Kaggle',
        'Model':      name,
        'Accuracy':   round(accuracy_score(y_test, y_pred), 4),
        'Precision':  round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall':     round(recall_score(y_test, y_pred, zero_division=0), 4),
        'F1':         round(f1_score(y_test, y_pred, zero_division=0), 4),
        'FPR':        round(fpr, 4),
        'Inference_ms_per_email': round(elapsed / len(y_test) * 1000, 6),
    })

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legitimate', 'Phishing'])
    fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f'Confusion Matrix — {name}')
    plt.tight_layout()
    plt.savefig(figures_dir / f"kaggle_cm_{name.lower().replace(' ', '_')}.png", dpi=150)
    plt.show()

## 8. Per-Source Breakdown
Evaluate each model separately on human-generated and LLM-generated emails.

In [ ]:
# Create boolean masks to filter test rows by source
human_mask = source_test.values == 'human-generated'
llm_mask   = source_test.values == 'llm-generated'

for name, y_pred in all_predictions.items():
    print(f"\n{'='*55}")
    print(f'  {name}')
    print(f"{'='*55}")

    print('  >> Human-generated emails:')
    print(classification_report(
        y_test[human_mask], y_pred[human_mask],
        target_names=['Legitimate (0)', 'Phishing (1)'],
        zero_division=0
    ))

    print('  >> LLM-generated emails:')
    print(classification_report(
        y_test[llm_mask], y_pred[llm_mask],
        target_names=['Legitimate (0)', 'Phishing (1)'],
        zero_division=0
    ))

## 9. Results Summary
Compile all model results into a single table and save to CSV.

In [ ]:
results_df = pd.DataFrame(all_results)
display(results_df)

results_metrics_dir = Path('../results/metrics')
results_metrics_dir.mkdir(parents=True, exist_ok=True)
results_df.to_csv(results_metrics_dir / 'kaggle_classical_baselines_results.csv', index=False)
print('Saved to:', results_metrics_dir / 'kaggle_classical_baselines_results.csv')